In [67]:
import pandas as pd
import numpy as np
import glob
import rioxarray
import re 

loc_coords = {
    "US-Ne1": (41.1651, -96.4766),
    "US-Ne2": (41.1649, -96.4701),
    "US-Ne3": (41.1797, -96.4397),
    "US-Var": (38.4133, -120.9508),
    "US-GLE": (41.3665, -106.2399),
    "US-Me2": (44.4526, -121.5589),
    'US-NR1': (40.0329, -105.5464),
    'US-ARM': (36.6058, -97.4888),
    'US-SRG': (31.7894, -110.8277),
    'US-Wkg': (31.7365, -109.9419),
    'US-WCr': (45.8059, -90.0799),
    'US-SRM': (31.8214, -110.8661),
    'US-CMW': (31.6637, -110.1777)
    }
sites = list(loc_coords.keys())
print(sites)

dir = '/Users/claireleeb/research/'
files = glob.glob(dir + '*.tif')

def extract_date(f):
    match = re.search(r'(\d{4})\.M(\d{2})', f)
    if match:
        return int(match.group(1)), int(match.group(2))
    return (9999, 99)  

files_sorted = sorted(files, key=extract_date)


sif_values = np.zeros((len(files_sorted), len(loc_coords)))

for i, filename in enumerate(files_sorted):
    ds = rioxarray.open_rasterio(filename)
    for j, name in enumerate(sites):
        coords = loc_coords[name]
        lat = coords[0]
        lon = coords[1]
        dt = ds.sel(x=lon, y=lat, method="nearest").values[0]
        sif_values[i, j] = dt * 0.0001  # scale factor



dates = [f"{y}-{m:02d}" for (y, m) in map(extract_date, files_sorted)]
df = pd.DataFrame(sif_values, columns= sites)
df['Date'] = pd.to_datetime(dates)
df = df.set_index('Date').sort_index()

print(df)
df.to_csv('GOSIF_timeseries_allsites.csv')

['US-Ne1', 'US-Ne2', 'US-Ne3', 'US-Var', 'US-GLE', 'US-Me2', 'US-NR1', 'US-ARM', 'US-SRG', 'US-Wkg', 'US-WCr', 'US-SRM', 'US-CMW']
            US-Ne1  US-Ne2  US-Ne3  US-Var  US-GLE  US-Me2  US-NR1  US-ARM  \
Date                                                                         
2002-07-01  0.3211  0.3211  0.3158  0.0818  0.1689  0.1894  0.1484  0.1317   
2002-08-01  0.2888  0.2888  0.3114  0.0658  0.1218  0.1626  0.1116  0.1169   
2014-07-01  0.4374  0.4374  0.4560  0.0849  0.1571  0.2095  0.1636  0.2109   

            US-SRG  US-Wkg  US-WCr  US-SRM  US-CMW  
Date                                                
2002-07-01  0.0426  0.0192  0.5491  0.0211  0.0253  
2002-08-01  0.1075  0.0646  0.4675  0.0440  0.0713  
2014-07-01  0.0811  0.0307  0.5602  0.0469  0.0371  
